In [135]:
import pandas as pd
import numpy as np
import os
import pandas_gbq
from google.cloud import bigquery
import glob
import openpyxl

Chamando as bases

In [136]:
mppm18 = pd.read_excel('g:\\Drives compartilhados\\República.org\\02. Áreas\\Dados e Conhecimento\\460 - Estudos e Infográficos\\07 - Mulheres no serviço público\\2026\\Dados\\bases brutas\\MUNIC\\munic_2018.xlsx', sheet_name='Política para mulheres')
mppm18

,Cod Municipio,MPPM01,MPPM02,MPPM031,MPPM032,MPPM033,MPPM034,MPPM035,MPPM036,MPPM04,...,MPPM246,MPPM247,MPPM248,MPPM249,MPPM2410,MPPM2411,MPPM25,MPPM251,MPPM252,MPPM26
0,1100015,Não,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,Não,-,-,Não
1,1100023,Sim,Setor subordinado a outra secretaria,Sim,Não,Não,Não,Não,Não,secretaria municipal de desenvolvimento social,...,Sim,Sim,Não,Sim,Sim,-,Sim,Não,-,Sim
2,1100031,Não,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,Não,-,-,Não
3,1100049,Não,-,-,-,-,-,-,-,-,...,Não,Não,Não,Sim,Não,-,Não,-,-,Não
4,1100056,Não,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,Não,-,-,Não
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5565,5222005,Sim,Setor subordinado a outra secretaria,Sim,Não,Não,Não,Não,Não,SECRETARIA MUNICIPAL DE ASSISTÊNCIA SOCIAL,...,-,-,-,-,-,-,Não,-,-,Não
5566,5222054,Não,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,Não,-,-,Não
5567,5222203,Não,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,Não,-,-,Não
5568,5222302,Não,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,Não,-,-,Não


In [168]:
uf = pd.read_excel('g:\\Drives compartilhados\\República.org\\02. Áreas\\Dados e Conhecimento\\460 - Estudos e Infográficos\\07 - Mulheres no serviço público\\2026\\Dados\\bases brutas\\MUNIC\\munic_2018.xlsx', sheet_name = 'Variáveis externas', usecols=[0,2,4]) # Pegando nome e codigo das UF
uf

,Cod Municipio,COD UF,NOME MUNIC
0,1100015,11,Alta Floresta D'Oeste
1,1100023,11,Ariquemes
2,1100031,11,Cabixi
3,1100049,11,Cacoal
4,1100056,11,Cerejeiras
...,...,...,...
5565,5222005,52,Vianópolis
5566,5222054,52,Vicentinópolis
5567,5222203,52,Vila Boa
5568,5222302,52,Vila Propício


In [142]:
mppm18 = pd.read_excel('g:\\Drives compartilhados\\República.org\\02. Áreas\\Dados e Conhecimento\\460 - Estudos e Infográficos\\07 - Mulheres no serviço público\\2026\\Dados\\bases brutas\\MUNIC\\munic_2018.xlsx', sheet_name='Política para mulheres', usecols=['Cod Municipio','MPPM01','MPPM02','MPPM05', 'MPPM06', 'MPPM07', 'MPPM08'])
mppm18['ano'] = 2018
uf = pd.read_excel('g:\\Drives compartilhados\\República.org\\02. Áreas\\Dados e Conhecimento\\460 - Estudos e Infográficos\\07 - Mulheres no serviço público\\2026\\Dados\\bases brutas\\MUNIC\\munic_2018.xlsx', sheet_name = 'Variáveis externas', usecols=[0,2,4]) # Pegando nome e codigo das UF
mppm18 = mppm18.merge(uf, right_on='Cod Municipio',left_on='Cod Municipio') # Juntando os dataframes, adicionando sigla e nome das UFs
mppm18['NOME MUNIC'] = mppm18['NOME MUNIC'].str.title()
mppm18= mppm18.rename(columns={'COD UF':'cod_uf',
                             'Cod Municipio':'id_municipio',
                             'NOME MUNIC':'nome_municipio',
                             'MPPM02':'caracterizacao_orgao_gestor',
                             'MPPM05':'genero',
                             'MPPM06':'idade',
                             'MPPM07':'cor_raca',
                             'MPPM08':'grau_instrucao'}) 
mppm18 = mppm18[['ano','cod_uf','id_municipio','nome_municipio','caracterizacao_orgao_gestor','genero','idade','cor_raca','grau_instrucao']]
cols = ['caracterizacao_orgao_gestor','genero', 'grau_instrucao','cor_raca']
for col in cols:
    mppm18[col]= mppm18[col].str.replace('-','Sem dados')
    mppm18[col]= mppm18[col].str.replace('Não informou','Sem dados')
    mppm18[col]= mppm18[col].str.replace('Recusa','Sem dados')
mppm18['idade']=np.where(mppm18['idade']=='-',np.nan,mppm18['idade']) 
mppm18['idade']=np.where(mppm18['idade']=='Não informou',np.nan,mppm18['idade']) 
mppm18['idade']=np.where(mppm18['idade']=='Recusa',np.nan,mppm18['idade'])  
mppm18['idade']=np.where(mppm18['idade']=='(*) Não soube informar',np.nan,mppm18['idade'])  
mppm18['idade'] =pd.to_numeric(mppm18['idade'])
mppm18

,ano,cod_uf,id_municipio,nome_municipio,caracterizacao_orgao_gestor,genero,idade,cor_raca,grau_instrucao
0,2018,11,1100015,Alta Floresta D'Oeste,Sem dados,Sem dados,NaN,Sem dados,Sem dados
1,2018,11,1100023,Ariquemes,Setor subordinado a outra secretaria,Feminino,34.0,Parda,Especialização
2,2018,11,1100031,Cabixi,Sem dados,Sem dados,NaN,Sem dados,Sem dados
3,2018,11,1100049,Cacoal,Sem dados,Sem dados,NaN,Sem dados,Sem dados
4,2018,11,1100056,Cerejeiras,Sem dados,Sem dados,NaN,Sem dados,Sem dados
...,...,...,...,...,...,...,...,...,...
5565,2018,52,5222005,Vianópolis,Setor subordinado a outra secretaria,Feminino,40.0,Branca,Especialização
5566,2018,52,5222054,Vicentinópolis,Sem dados,Sem dados,NaN,Sem dados,Sem dados
5567,2018,52,5222203,Vila Boa,Sem dados,Sem dados,NaN,Sem dados,Sem dados
5568,2018,52,5222302,Vila Propício,Sem dados,Sem dados,NaN,Sem dados,Sem dados


In [138]:
mppm18['idade'].value_counts() #- , (*) Não soube informar, Recusa, Não informou
mppm18['cor_raca'].value_counts() #Não informou, Recusa, -  
mppm18['grau_instrucao'].value_counts() #Não informou, Recusa, -   
mppm18['genero'].value_counts() #Não informou, Recusa, -   

genero
Sem dados    4461
Feminino     1004
Masculino     105
Name: count, dtype: int64

In [144]:
mppm23 = pd.read_excel('g:\\Drives compartilhados\\República.org\\02. Áreas\\Dados e Conhecimento\\460 - Estudos e Infográficos\\07 - Mulheres no serviço público\\2026\\Dados\\bases brutas\\MUNIC\\munic_2023.xlsx', sheet_name='Política para Mulheres', usecols=['CodMun','Cod UF','Mun','MPPM02','MPPM05', 'MPPM06', 'MPPM07', 'MPPM08'])
mppm23['ano'] = 2023
mppm23['Mun'] = mppm23['Mun'].str.title()
mppm23= mppm23.rename(columns={'Cod UF':'cod_uf',
                             'CodMun':'id_municipio',
                             'Mun':'nome_municipio',
                             'MPPM02':'caracterizacao_orgao_gestor',
                             'MPPM05':'genero',
                             'MPPM06':'idade',
                             'MPPM07':'cor_raca',
                             'MPPM08':'grau_instrucao'}) 
mppm23 = mppm23[['ano','cod_uf','id_municipio','nome_municipio','caracterizacao_orgao_gestor','genero','idade','cor_raca','grau_instrucao']]
cols = ['caracterizacao_orgao_gestor','genero', 'grau_instrucao','cor_raca']
for col in cols:
    mppm23[col]= mppm23[col].str.replace('(**) Sem gestor','Sem dados')
    mppm23[col]= mppm23[col].str.replace('Não informou','Sem dados')
    mppm23[col]= mppm23[col].str.replace('-','Sem dados')
mppm23['idade']=np.where(mppm23['idade']=='-',np.nan,mppm23['idade']) 
mppm23['idade']=np.where(mppm23['idade']=='Não informou',np.nan,mppm23['idade']) 
mppm23['idade']=np.where(mppm23['idade']=='(**) Sem gestor',np.nan,mppm23['idade'])  
mppm23['idade']=np.where(mppm23['idade']=='-',np.nan,mppm23['idade'])  
mppm23 

,ano,cod_uf,id_municipio,nome_municipio,caracterizacao_orgao_gestor,genero,idade,cor_raca,grau_instrucao
0,2023,11,1100015,Alta Floresta Doeste,Sem dados,Sem dados,NaN,Sem dados,Sem dados
1,2023,11,1100023,Ariquemes,Setor subordinado a outra secretaria,Feminino,36,Branca,Especialização
2,2023,11,1100031,Cabixi,Sem dados,Sem dados,NaN,Sem dados,Sem dados
3,2023,11,1100049,Cacoal,Sem dados,Sem dados,NaN,Sem dados,Sem dados
4,2023,11,1100056,Cerejeiras,Setor subordinado a outra secretaria,Masculino,28,Parda,Especialização
...,...,...,...,...,...,...,...,...,...
5565,2023,52,5222005,Vianópolis,Setor subordinado a outra secretaria,Feminino,35,Branca,Ensino superior incompleto
5566,2023,52,5222054,Vicentinópolis,Sem dados,Sem dados,NaN,Sem dados,Sem dados
5567,2023,52,5222203,Vila Boa,Sem dados,Sem dados,NaN,Sem dados,Sem dados
5568,2023,52,5222302,Vila Propício,Sem dados,Sem dados,NaN,Sem dados,Sem dados


In [145]:
mppm = pd.concat([mppm18,mppm23], ignore_index=True)
mppm

,ano,cod_uf,id_municipio,nome_municipio,caracterizacao_orgao_gestor,genero,idade,cor_raca,grau_instrucao
0,2018,11,1100015,Alta Floresta D'Oeste,Sem dados,Sem dados,NaN,Sem dados,Sem dados
1,2018,11,1100023,Ariquemes,Setor subordinado a outra secretaria,Feminino,34.0,Parda,Especialização
2,2018,11,1100031,Cabixi,Sem dados,Sem dados,NaN,Sem dados,Sem dados
3,2018,11,1100049,Cacoal,Sem dados,Sem dados,NaN,Sem dados,Sem dados
4,2018,11,1100056,Cerejeiras,Sem dados,Sem dados,NaN,Sem dados,Sem dados
...,...,...,...,...,...,...,...,...,...
11135,2023,52,5222005,Vianópolis,Setor subordinado a outra secretaria,Feminino,35,Branca,Ensino superior incompleto
11136,2023,52,5222054,Vicentinópolis,Sem dados,Sem dados,NaN,Sem dados,Sem dados
11137,2023,52,5222203,Vila Boa,Sem dados,Sem dados,NaN,Sem dados,Sem dados
11138,2023,52,5222302,Vila Propício,Sem dados,Sem dados,NaN,Sem dados,Sem dados


In [146]:
# criando dicionário
dict_esco = {'Sem dados':'Sem dados',
            'Especialização':'Até Pós Graduação ou Mestrado',
            'Ensino superior completo':'Até Ensino Superior Completo',
            'Ensino superior incompleto':'Até Ensino Superior Completo',
            'Ensino fundamental (1º Grau) incompleto':'Até Ensino Médio',
            'Ensino médio (2º Grau) completo':'Até Ensino Médio',
            'Ensino médio (2º Grau) incompleto':'Até Ensino Médio',
            'Mestrado':'Até Pós Graduação ou Mestrado',
            'Ensino fundamental (1º Grau) completo':'Até Ensino Médio',
            'Doutorado':'Até Doutorado',
            'Ensino fundamental ( 1º Grau) completo':'Até Ensino Médio'}


In [148]:
mppm['grau_instrucao'].unique()

array(['Sem dados', 'Especialização', 'Ensino superior completo',
       'Ensino superior incompleto',
       'Ensino fundamental (1º Grau) incompleto',
       'Ensino médio (2º Grau) completo',
       'Ensino médio (2º Grau) incompleto', 'Mestrado',
       'Ensino fundamental (1º Grau) completo', 'Doutorado',
       'Ensino fundamental ( 1º Grau) completo'], dtype=object)

In [149]:
mppm = mppm.replace({'grau_instrucao':dict_esco}) #substituindo valores para padronizar
mppm

,ano,cod_uf,id_municipio,nome_municipio,caracterizacao_orgao_gestor,genero,idade,cor_raca,grau_instrucao
0,2018,11,1100015,Alta Floresta D'Oeste,Sem dados,Sem dados,NaN,Sem dados,Sem dados
1,2018,11,1100023,Ariquemes,Setor subordinado a outra secretaria,Feminino,34.0,Parda,Até Pós Graduação ou Mestrado
2,2018,11,1100031,Cabixi,Sem dados,Sem dados,NaN,Sem dados,Sem dados
3,2018,11,1100049,Cacoal,Sem dados,Sem dados,NaN,Sem dados,Sem dados
4,2018,11,1100056,Cerejeiras,Sem dados,Sem dados,NaN,Sem dados,Sem dados
...,...,...,...,...,...,...,...,...,...
11135,2023,52,5222005,Vianópolis,Setor subordinado a outra secretaria,Feminino,35,Branca,Até Ensino Superior Completo
11136,2023,52,5222054,Vicentinópolis,Sem dados,Sem dados,NaN,Sem dados,Sem dados
11137,2023,52,5222203,Vila Boa,Sem dados,Sem dados,NaN,Sem dados,Sem dados
11138,2023,52,5222302,Vila Propício,Sem dados,Sem dados,NaN,Sem dados,Sem dados


In [150]:
mppm['idade'] =pd.to_numeric(mppm['idade'])

In [151]:
limites = [0, 29, 49, 64, 100] #criando uma nova coluna (faixa_etaria) com base na coluna 'idade'
categorias = ['Entre 18-29', 'Entre 30-49', 'Entre 50-65', 'Acima de 65']

mppm['faixa_etaria'] = pd.cut(mppm['idade'], bins=limites, labels=categorias)


In [ ]:
mppm[mppm['idade']==49]['faixa_etaria'].value_counts()

In [126]:
mppm.columns

Index(['ano', 'cod_uf', 'cod_municipio', 'nome_municipio',
       'caracterizacao_orgao_gestor', 'genero', 'faixa_etaria', 'cor_raca',
       'grau_instrucao'],
      dtype='object')

In [157]:
mppm['grau_instrucao'].unique()

array(['Sem dados', 'Até Pós Graduação ou Mestrado',
       'Até Ensino Superior Completo', 'Até Ensino Médio',
       'Até Doutorado'], dtype=object)

Subindo para o GBQ

In [158]:
client = bigquery.Client()
dataset_ref = client.dataset('cargos_lideranca')

In [159]:
mppm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11140 entries, 0 to 11139
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   ano                          11140 non-null  int64   
 1   cod_uf                       11140 non-null  int64   
 2   id_municipio                 11140 non-null  int64   
 3   nome_municipio               11140 non-null  object  
 4   caracterizacao_orgao_gestor  11140 non-null  object  
 5   genero                       11140 non-null  object  
 6   idade                        2839 non-null   float64 
 7   cor_raca                     11140 non-null  object  
 8   grau_instrucao               11140 non-null  object  
 9   faixa_etaria                 2839 non-null   category
dtypes: category(1), float64(1), int64(3), object(5)
memory usage: 794.5+ KB


In [ ]:
mppm = mppm[['ano', 'cod_uf','id_municipio','nome_municipio','caracterizacao_orgao_gestor','genero', 'faixa_etaria', 'cor_raca', 'grau_instrucao']]
mppm

,ano,cod_uf,cod_municipio,nome_municipio,caracterizacao_orgao_gestor,genero,faixa_etaria,cor_raca,grau_instrucao
0,2018,11,1100015,Alta Floresta D'Oeste,Sem dados,Sem dados,NaN,Sem dados,Sem dados
1,2018,11,1100023,Ariquemes,Setor subordinado a outra secretaria,Feminino,Entre 30-49,Parda,Até Pós Graduação ou Mestrado
2,2018,11,1100031,Cabixi,Sem dados,Sem dados,NaN,Sem dados,Sem dados
3,2018,11,1100049,Cacoal,Sem dados,Sem dados,NaN,Sem dados,Sem dados
4,2018,11,1100056,Cerejeiras,Sem dados,Sem dados,NaN,Sem dados,Sem dados
...,...,...,...,...,...,...,...,...,...
11135,2023,52,5222005,Vianópolis,Setor subordinado a outra secretaria,Feminino,Entre 30-49,Branca,Até Ensino Superior Completo
11136,2023,52,5222054,Vicentinópolis,Sem dados,Sem dados,NaN,Sem dados,Sem dados
11137,2023,52,5222203,Vila Boa,Sem dados,Sem dados,NaN,Sem dados,Sem dados
11138,2023,52,5222302,Vila Propício,Sem dados,Sem dados,NaN,Sem dados,Sem dados


In [160]:
mppm.dtypes

ano                               int64
cod_uf                            int64
id_municipio                      int64
nome_municipio                   object
caracterizacao_orgao_gestor      object
genero                           object
idade                           float64
cor_raca                         object
grau_instrucao                   object
faixa_etaria                   category
dtype: object

In [167]:
for col in mppm.columns:
    print(mppm[col].name)
    print(mppm[col].unique())
    print(mppm[col].dtype)
    print ('-----------------------------')

ano
[2018 2023]
int64
-----------------------------
cod_uf
[11 12 13 14 15 16 17 21 22 23 24 25 26 27 28 29 31 32 33 35 41 42 43 50
 51 52 53]
int64
-----------------------------
id_municipio
[1100015 1100023 1100031 ... 5222203 5222302 5300108]
int64
-----------------------------
nome_municipio
["Alta Floresta D'Oeste" 'Ariquemes' 'Cabixi' ... 'Mirassol Doeste'
 'São João Daliança' 'Sítio Dabadia']
object
-----------------------------
caracterizacao_orgao_gestor
['Sem dados' 'Setor subordinado a outra secretaria'
 'Secretaria em conjunto com outras políticas setoriais'
 'Secretaria exclusiva'
 'Setor subordinado diretamente à chefia do Executivo'
 'Órgão da administração indireta']
object
-----------------------------
genero
['Sem dados' 'Feminino' 'Masculino']
object
-----------------------------
idade
[nan 34. 56. 60. 41. 37. 42. 36. 26. 43. 63. 29. 25. 35. 38. 52. 54. 55.
 44. 59. 46. 51. 30. 47. 21. 45. 50. 64. 32. 65. 39. 40. 49. 48. 33. 31.
 77. 22. 66. 69. 74. 28. 53. 67. 24. 5

In [161]:
schema=[bigquery.SchemaField('ano','INTEGER',description='Ano referente a informação'),
        bigquery.SchemaField('cod_uf','INTEGER',description='Sigla da UF'),
        bigquery.SchemaField('id_municipio','INTEGER',description='Código do IBGE da UF'),
        bigquery.SchemaField('nome_municipio','STRING',description='Caracterização do órgão no qual o gestor está'), 
        bigquery.SchemaField('caracterizacao_orgao_gestor','STRING',description='Nome da UF'), 
        bigquery.SchemaField('genero','STRING',description='Gênero autodeclarado ou não'),
        bigquery.SchemaField('faixa_etaria','STRING',description='Faixa etária da observação'),
        bigquery.SchemaField('cor_raca','STRING',description='Raça/cor da pessoa observada'),
        bigquery.SchemaField('grau_instrucao','STRING',description='Escolaridade da pessoa ou do vínculo observado com detalhamento na pós-graduação')
        ]


In [165]:
table_ref = dataset_ref.table('MUNIC_perfil_gestor_politica_mulheres_tipo_orgao')
job_config = bigquery.LoadJobConfig(schema=schema)
job = client.load_table_from_dataframe(mppm,table_ref, job_config=job_config)
job.result() 

LoadJob<project=repositoriodedadosgpsp, location=US, id=579efad5-14b7-48a9-afa2-ae44849a8f24>